# 5 参数初始化

学习目标
- 知道参数初始化的重要性
- 知道常见的参数初始化方法
- 能够使用 PyTorch 实现参数初始化

网络参数初始化是训练深度学习模型的第一步，选择恰当的初始化方法十分重要。

如果权重初始化得太小，从某层经过激活函数得到的输出会非常小，多层叠加之后数值会越来越接近 0，反向传播时梯度也会随之越来越小，最终导致**梯度消失**，网络在浅层的参数几乎无法更新。

如果权重初始化得太大，数据经过每一层之后会被放大，多层叠加之后数值会变得越来越大（甚至溢出），反向传播时梯度也会随之越来越大，最终导致**梯度爆炸**，网络的训练过程将会非常不稳定。

合适的权重初始化方法，会使得各层的输出在经过多层网络后依然保持比较好的尺度（不会过大或过小），从而可以有效地避免梯度消失和梯度爆炸的问题。

## 1. 固定值初始化

### 1.1 全 0 初始化

将神经网络中的所有权重参数初始化为 0。

这种初始化方法存在严重的问题：在反向传播过程中，由于每一层的权重值都相同（都是 0），所以每个神经元在反向传播时计算出的梯度值也相同，这就意味着参数更新也是相同的，这就使得隐藏层神经元的对称性无法被破坏，最终网络退化为等价于一个线性模型，失去了深度网络的意义。这种现象被称为**对称权重现象**。

### 1.2 全 1 初始化

同全 0 初始化一样，全 1 初始化也会使得隐藏层神经元的对称性无法被打破，每个神经元在迭代过程中表现完全一致，所以也不可取。

### 1.3 任意常数初始化

将所有参数初始化为某个相同的常数（如 0.5），同样会导致对称权重问题，每层神经元的输出值都相同，参数更新也相同，使得网络的学习能力大打折扣。

**结论：固定值初始化方法都存在对称权重问题，在实践中很少使用。**

## 2. 随机初始化

随机初始化是将参数初始化为随机的较小的值，这是最常用的初始化方法之一。常见的随机初始化方式有：均匀分布初始化、正态分布初始化等。

### 2.1 均匀分布初始化

权重参数从区间 $[-r, r]$ 均匀分布中随机采样，其中 $r$ 是一个比较小的数，如 0.05。

$$w \sim U(-r, r)$$

### 2.2 正态分布初始化

权重参数从均值为 0，标准差为某个值（如 0.01）的高斯分布中随机采样。

$$w \sim N(0, \sigma^2)$$

随机初始化打破了对称权重问题，但如果初始化的值过大或过小，仍然容易导致梯度消失或梯度爆炸的问题。所以，随着网络层数的增多，我们一般不直接使用简单的随机初始化方法，而是使用根据网络结构（输入输出神经元个数）计算得到的初始化方法，如下面要介绍的 Xavier 初始化和 Kaiming 初始化。

## 3. Xavier 初始化（Glorot 初始化）

Xavier 初始化是 Glorot 和 Bengio 在 2010 年提出的，其核心思想是：**让每一层输出的方差尽量等于输入的方差**，从而保持前向传播和反向传播过程中数据的尺度不变，避免梯度消失和梯度爆炸。

设某一层有 $n_{in}$ 个输入神经元和 $n_{out}$ 个输出神经元，Xavier 初始化的公式如下：

### 3.1 均匀分布版本（Xavier Uniform）

$$w \sim U\left(-\sqrt{\frac{6}{n_{in} + n_{out}}},\ \sqrt{\frac{6}{n_{in} + n_{out}}}\right)$$

### 3.2 正态分布版本（Xavier Normal）

$$w \sim N\left(0,\ \frac{2}{n_{in} + n_{out}}\right)$$

Xavier 初始化是在**激活函数为线性或 sigmoid/tanh** 的前提下推导出来的，因为这些激活函数在 0 附近近似线性。

> 注意：**Xavier 初始化不适合 ReLU 激活函数**，因为 ReLU 会将负数截断为 0，实际上有效的神经元只有一半，方差会变为原来的一半，随着层数增加，数值会越来越小。

在 PyTorch 中可以使用 `nn.init.xavier_uniform_` 和 `nn.init.xavier_normal_` 来实现。

## 4. Kaiming 初始化（He 初始化）

Kaiming 初始化是何恺明等人在 2015 年提出的，专门针对 **ReLU 及其变体**激活函数设计。

ReLU 激活函数会将负数截断为 0，因此实际上只有约一半的神经元处于激活状态，这导致 Xavier 初始化的方差假设不再成立。Kaiming 初始化在推导时考虑了这个因素，对方差做了修正：

### 4.1 均匀分布版本（Kaiming Uniform）

$$w \sim U\left(-\sqrt{\frac{6}{n_{in}}},\ \sqrt{\frac{6}{n_{in}}}\right)$$

### 4.2 正态分布版本（Kaiming Normal）

$$w \sim N\left(0,\ \frac{2}{n_{in}}\right)$$

其中 $n_{in}$ 是当前层的输入神经元个数。

| 激活函数 | 推荐初始化方法 |
| :--- | :--- |
| sigmoid / tanh | Xavier 初始化 |
| ReLU / Leaky ReLU | Kaiming 初始化 |
| 线性激活 | Xavier 初始化 |

**PyTorch 默认对 `nn.Linear` 使用 Kaiming Uniform 初始化。**

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt


# ============================================================
# 演示各种初始化方法对权重分布的影响
# ============================================================

def init_demo():
    """对比各种初始化方法下，权重的分布情况"""
    layer = nn.Linear(256, 256, bias=False)
    init_methods = {
        '全0初始化':       lambda w: nn.init.constant_(w, 0),
        '随机正态(σ=0.01)': lambda w: nn.init.normal_(w, mean=0, std=0.01),
        'Xavier Uniform':  lambda w: nn.init.xavier_uniform_(w),
        'Xavier Normal':   lambda w: nn.init.xavier_normal_(w),
        'Kaiming Uniform': lambda w: nn.init.kaiming_uniform_(w, nonlinearity='relu'),
        'Kaiming Normal':  lambda w: nn.init.kaiming_normal_(w, nonlinearity='relu'),
    }

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()

    for ax, (name, init_fn) in zip(axes, init_methods.items()):
        init_fn(layer.weight)
        weights = layer.weight.data.numpy().flatten()
        ax.hist(weights, bins=80, color='steelblue', edgecolor='none')
        ax.set_title(name)
        ax.set_xlabel('权重值')
        ax.set_ylabel('频次')
        ax.grid(True, alpha=0.3)
        std_val = weights.std()
        ax.set_title(f'{name}\n(σ={std_val:.4f})')

    plt.tight_layout()
    plt.suptitle('各种参数初始化方法对权重分布的影响', y=1.02, fontsize=14)
    plt.show()


if __name__ == '__main__':
    init_demo()

## 5. PyTorch 中的参数初始化 API

PyTorch 在 `torch.nn.init` 模块中提供了多种参数初始化方法，常用 API 如下：

| 初始化方法 | PyTorch API | 适用场景 |
| :--- | :--- | :--- |
| 全 0 初始化 | `nn.init.constant_(tensor, 0)` | bias 常用 |
| 全 1 初始化 | `nn.init.constant_(tensor, 1)` | BatchNorm γ |
| 任意常数初始化 | `nn.init.constant_(tensor, val)` | 特殊场景 |
| 均匀分布随机初始化 | `nn.init.uniform_(tensor, a, b)` | 简单随机 |
| 正态分布随机初始化 | `nn.init.normal_(tensor, mean, std)` | 简单随机 |
| Xavier Uniform | `nn.init.xavier_uniform_(tensor)` | sigmoid/tanh |
| Xavier Normal | `nn.init.xavier_normal_(tensor)` | sigmoid/tanh |
| Kaiming Uniform | `nn.init.kaiming_uniform_(tensor, nonlinearity='relu')` | ReLU（PyTorch 默认） |
| Kaiming Normal | `nn.init.kaiming_normal_(tensor, nonlinearity='relu')` | ReLU |
| 单位矩阵初始化 | `nn.init.eye_(tensor)` | 方阵，RNN 场景 |
| 正交初始化 | `nn.init.orthogonal_(tensor)` | RNN/LSTM |

> **说明**：PyTorch 中带下划线 `_` 的函数为原地操作（in-place），直接修改传入的 tensor，不会产生新对象。

## 6. 在网络中自定义参数初始化

在实际项目中，通常通过定义一个初始化函数，遍历网络的所有模块，并根据层的类型选择对应的初始化方法。示例如下：

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

import torch
import torch.nn as nn


# ============================================================
# 自定义网络参数初始化示例
# ============================================================

class MyNet(nn.Module):

    def __init__(self):
        super(MyNet, self).__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

        # 应用自定义初始化
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                # 使用 Kaiming Normal 初始化权重（适合 ReLU）
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                # 将 bias 初始化为 0
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


if __name__ == '__main__':
    net = MyNet()

    # 查看各层的初始化结果
    for name, param in net.named_parameters():
        print(f'{name:20s}  shape={str(param.shape):20s}  mean={param.data.mean():.4f}  std={param.data.std():.4f}')


## 7. 总结

| 初始化方法 | 核心思想 | 优点 | 缺点 |
| :--- | :--- | :--- | :--- |
| 全 0 / 全 1 / 常数 | 固定值 | 简单 | 对称权重问题，网络退化为线性 |
| 随机正态 / 均匀 | 随机小值 | 打破对称性 | 过大/小仍导致梯度问题 |
| Xavier | 方差与输入输出神经元数相关 | 适合 sigmoid/tanh，保持方差稳定 | 不适合 ReLU |
| Kaiming | 考虑 ReLU 截断修正方差 | 专为 ReLU 设计，更稳定 | PyTorch 默认，实践效果好 |

**实践建议：**

1. 使用 ReLU 激活函数时，优先选择 **Kaiming 初始化**（PyTorch 对 `nn.Linear` 的默认行为）
2. 使用 sigmoid / tanh 激活函数时，优先选择 **Xavier 初始化**
3. bias 通常初始化为 **0**
4. BatchNorm 层的 weight（γ）初始化为 **1**，bias（β）初始化为 **0**
5. 避免使用固定值（全 0 / 全 1）初始化权重，会导致对称权重问题

参数初始化是训练深度网络的基础，结合上一章的反向传播算法，合适的初始化可以让网络从一开始就处于有利的训练起点，显著加快收敛速度并提升最终性能。